# Fruitfly Rollout Analysis

This notebook demonstrates how to:
1. Download a pre-trained fruitfly model and reference data from HuggingFace
2. Load the model checkpoint and configure the environment
3. Generate a rollout (simulation) of fruitfly behavior
4. Render and save the rollout as a video
5. Create a timelapse visualization of key frames
6. Analyze joint kinematics by comparing reference motion capture data with the simulated rollout
7. Visualize neural network activations using PCA

The analysis helps validate how well the trained policy reproduces natural fruitfly movement patterns by visualizing both the rendered motion and quantitative joint angle comparisons.

## Imports & Setup

First, we configure the rendering backend and import necessary libraries.

In [ ]:
# Standard library imports
import os
import sys
import warnings

# Configure warnings
warnings.filterwarnings('ignore')

# Configure environment variables BEFORE importing JAX
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

# Set rendering backend based on platform
# macOS: use glfw (native OpenGL)
# Linux: use osmesa (software rendering) or egl (headless GPU)
if sys.platform == "darwin":
    os.environ["MUJOCO_GL"] = "glfw"
    print("Using macOS with GLFW rendering")
else:
    # Change this to egl if GPU is available on Linux, if not change to osmesa for software rendering
    os.environ["MUJOCO_GL"] = "egl"
    print("Using Linux with egl rendering")

# Third-party imports
import jax
from jax import numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mediapy as media
import mujoco
import huggingface_hub as hf_hub
from pathlib import Path
from sklearn.decomposition import PCA
from matplotlib.patches import Patch

# Jupyter notebook magic commands
%matplotlib inline
%config InlineBackend.figure_format='retina'

# Logging configuration
from absl import logging as absl_logging
absl_logging.set_verbosity(absl_logging.ERROR)

# Local imports
from track_mjx.agent import checkpointing
from track_mjx.analysis import rollout, render


## Some useful helper functions

def get_joint_qpos(model, qposes, joint_name):
    """Get qpos value(s) for a specific joint by name.
    Args:
        model: The Mujoco model.
        qposes: Array of joint position, shape (n_frames, qpos_dim).
        joint_name: Name of the joint to extract.
    Returns:
        The qpos value(s) for the specified joint
    """
    joint_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, joint_name)
    if joint_id == -1:
        print(f"Joint '{joint_name}' not found!")
        return None

    qpos_addr = model.jnt_qposadr[joint_id]
    joint_type = model.jnt_type[joint_id]

    if joint_type == mujoco.mjtJoint.mjJNT_FREE:
        return qposes[:, qpos_addr : qpos_addr + 7]  # position + quaternion
    elif joint_type == mujoco.mjtJoint.mjJNT_HINGE:
        return qposes[:, qpos_addr]  # single angle
    elif joint_type == mujoco.mjtJoint.mjJNT_BALL:
        return qposes[:, qpos_addr : qpos_addr + 4]  # quaternion
    else:
        return qposes[:, qpos_addr]

## Download Model from HuggingFace

Download the pre-trained fruitfly model checkpoint from the MIMIC-MJX model repository. This checkpoint contains the trained policy weights and configuration.

In [ ]:
hf_checkpoint_path = "fruitfly"
model_local_dir = Path.cwd().parent / "model_checkpoints"
# Download model from model repo
model_download_dir = hf_hub.snapshot_download(
    repo_id="talmolab/MIMIC-MJX",
    repo_type="model",  # download from model repo
    allow_patterns=hf_checkpoint_path + "/*",  # path with model id
    local_dir=model_local_dir,
)
print(f"Downloaded model to {model_download_dir}")

## Download Reference Data

Download the fruitfly reference clips dataset that contains the motion capture data used for training and evaluation.

In [ ]:
hf_data_path = "data/fly/fly_reference_clip.h5"
# Download data from dataset repo
data_download_dir = hf_hub.hf_hub_download(
    repo_id="talmolab/MIMIC-MJX",
    repo_type="dataset",  # download from dataset repo
    filename=hf_data_path,  # dataset name
    local_dir=Path.cwd().parent,
)
print(f"Downloaded data to {data_download_dir}")

## Load Checkpoint and Configure

Load the checkpoint and update the configuration with the downloaded data path.

In [ ]:
# replace with your checkpoint path
ckpt_path = model_local_dir / hf_checkpoint_path

ckpt = checkpointing.load_checkpoint_for_eval(ckpt_path)
cfg = ckpt["cfg"]

# Update the data path to the downloaded data
cfg.data_path = Path(data_download_dir)

In [ ]:
# TEMP: add max_start_frames to cfg to support older checkpoints
cfg.env_config.env_args.max_start_frame = 40

## Create Environment and Rollout Generator

Set up the simulation environment, load the inference function from the trained policy, and create a rollout generator that will produce simulations.

In [ ]:
env = rollout.create_environment(cfg)
inference_fn = checkpointing.load_inference_fn(cfg, ckpt["policy"])
generate_rollout = rollout.create_rollout_generator(
    cfg,
    env,
    inference_fn,
    log_activations=True,
    log_metrics=False,
    log_sensor_data=False,
)

## Generate Rollout

Generate a single rollout using clip index 22 from the reference data. This will simulate the fruitfly's behavior based on the trained policy. Feel free to change the clip index to visualize different behaviors in this dataset.

> **Note:** The first time you run this cell, it may take up to 2 minutes to compile the model with JAX, depending on your hardware. Subsequent runs will be much faster.


In [ ]:
single_rollout = generate_rollout(clip_idx=22)

## Render and Save Video

Render the rollout as video frames and save it as an MP4 file. The video will be saved in the checkpoint directory.

In [ ]:
frames, realtime_framerate = render.render_rollout(
    cfg,
    single_rollout,
    height=480,
    width=640,
)

# save the video to disk
media.write_video(Path(ckpt_path) / "rollout_fly.mp4", frames, fps=realtime_framerate)
# slow down the video for visualization
media.show_video(frames, fps=realtime_framerate / 2)

## Create Timelapse Visualization

Sample every 100th frame from the video and display the first 4 frames in a single row. This provides a quick overview of the fruitfly's behavior progression throughout the rollout.

In [ ]:
# Sample every 50th frame and take only 4 frames
sampled_frames = frames[::100][:4]
num_frames = len(sampled_frames)

# Fixed grid dimensions: 1 row, 4 columns
cols = 4
rows = 1

# Create the figure and subplots
fig, axes = plt.subplots(rows, cols, figsize=(16, 4))
axes = axes.flatten()

# Plot each sampled frame
for idx, frame in enumerate(sampled_frames):
    axes[idx].imshow(frame)
    axes[idx].set_title(f"Frame {idx * 100}")
    axes[idx].axis("off")

# Hide any unused subplots
for idx in range(num_frames, len(axes)):
    axes[idx].axis("off")

plt.tight_layout()
plt.savefig(Path(ckpt_path) / "rollout_timelapse_fly.pdf", dpi=150, bbox_inches="tight")
plt.show()

## Visualize Rollout Kinematic Statistics

This section analyzes the joint kinematics by comparing the reference motion capture data (STAC Registration) with the simulated rollout (Track Replay). 

We'll plot the joint angles over time for key leg joints (femurs) to assess how accurately the trained policy reproduces the reference motion. The red shaded area highlights the differences between reference and simulated trajectories, providing a visual measure of tracking accuracy.

### Joint Angle Comparison

Extract joint positions (qpos) from both the reference and rollout data, then plot the joint angles over time. The visualization includes:

- **Orange line**: STAC Registration (ground truth motion capture)
- **Blue line**: Track Replay (simulated rollout from trained policy)
- **Red shaded area**: Magnitude of difference between reference and rollout

All subplots share the same y-axis scale for easier comparison across joints. Joint angles are displayed in degrees for intuitive interpretation.

In [ ]:
qposes_ref = single_rollout["qposes_ref"]
qposes_rollout = single_rollout["qposes_rollout"]

qposes_ref.shape, qposes_rollout.shape

In [ ]:
qposes_ref = single_rollout["qposes_ref"]
qposes_rollout = single_rollout["qposes_rollout"]
# Cap the length to the minimum of the two arrays
min_length = min(len(qposes_ref), len(qposes_rollout))
qposes_ref = qposes_ref[:min_length]
qposes_rollout = qposes_rollout[:min_length]
# Create subplots for joint angle comparison
# Create subplots for joint angle comparison
joint_names = [
    "femur_T1_left",
    # "femur_T2_left",
    "femur_T1_right",
    # "femur_T2_right",
    # "femur_T3_left",
    # "femur_T3_right",
]
joint_labels = [
    "Left\nFemur T1",
    # "Left\nFemur T2",
    "Right\nFemur T1",
    # "Right\nFemur T2",
    # "Left\nFemur T3",
    # "Right\nFemur T3",
]

# Set up the subplots
fig, axes = plt.subplots(len(joint_names), 1, figsize=(4, 3), sharex=True)

# First pass: collect all angle data to determine global y-axis limits
all_ref_angles = []
all_rollout_angles = []

for joint_name in joint_names:
    ref_angles = get_joint_qpos(env.sys.mj_model, qposes_ref, f"{joint_name}")
    rollout_angles = get_joint_qpos(env.sys.mj_model, qposes_rollout, f"{joint_name}")

    # Convert from radians to degrees
    ref_angles_deg = np.degrees(ref_angles)
    rollout_angles_deg = np.degrees(rollout_angles)

    all_ref_angles.extend(ref_angles_deg)
    all_rollout_angles.extend(rollout_angles_deg)

# Calculate global y-axis limits with some padding
all_angles = all_ref_angles + all_rollout_angles
y_min = np.min(all_angles) - 5  # Add 5 degree padding
y_max = np.max(all_angles) + 5  # Add 5 degree padding

# Plot each joint in its own subplot
for i, (joint_name, joint_label) in enumerate(zip(joint_names, joint_labels)):
    # Get joint angles for both reference and rollout
    ref_angles = get_joint_qpos(env.sys.mj_model, qposes_ref, f"{joint_name}")
    rollout_angles = get_joint_qpos(env.sys.mj_model, qposes_rollout, f"{joint_name}")

    # Convert from radians to degrees
    ref_angles_deg = np.degrees(ref_angles)
    rollout_angles_deg = np.degrees(rollout_angles)

    # Create time axis
    time_steps = np.arange(len(ref_angles))
    # Convert time steps to seconds (100 Hz = 0.01 seconds per step)
    time_seconds = time_steps * 0.01

    # Plot using seaborn style with solid lines for both
    sns.lineplot(
        x=time_seconds,
        y=ref_angles_deg,
        ax=axes[i],
        color="orange",
        label="STAC Registration",
        linewidth=2,
    )
    sns.lineplot(
        x=time_seconds,
        y=rollout_angles_deg,
        ax=axes[i],
        color="skyblue",
        label="Track Replay",
        linewidth=2,
    )

    # Highlight differences by filling the area between lines
    axes[i].fill_between(
        time_seconds,
        ref_angles_deg,
        rollout_angles_deg,
        alpha=0.2,
        color="red",
    )

    # Customize subplot
    axes[i].set_ylabel(joint_label)

    # Set consistent y-axis limits for all subplots
    axes[i].set_ylim(y_min, y_max)

    # Set informative y-axis ticks
    # Create ticks at nice intervals that include both positive and negative values
    tick_range = y_max - y_min
    if tick_range > 80:
        # For large ranges, use 25-degree intervals
        tick_interval = 25
    elif tick_range > 40:
        # For medium ranges, use 20-degree intervals
        tick_interval = 20
    else:
        # For small ranges, use 10-degree intervals
        tick_interval = 10

    # Generate ticks from the lowest multiple of tick_interval to the highest
    tick_start = int(np.floor(y_min / tick_interval)) * tick_interval
    tick_end = int(np.ceil(y_max / tick_interval)) * tick_interval
    y_ticks = np.arange(tick_start, tick_end + tick_interval, tick_interval)

    # Filter ticks to only show those within our y-limits
    y_ticks = y_ticks[(y_ticks >= y_min) & (y_ticks <= y_max)]
    axes[i].set_yticks(y_ticks)

    # Set informative x-axis ticks in seconds
    max_time = time_seconds[-1]
    if max_time <= 2:
        # For short sequences, use 0.5 second intervals
        x_tick_interval = 0.5
    elif max_time <= 5:
        # For medium sequences, use 1 second intervals
        x_tick_interval = 1.0
    else:
        # For longer sequences, use 2 second intervals
        x_tick_interval = 2.0

    x_ticks = np.arange(0, max_time, x_tick_interval)
    axes[i].set_xticks(x_ticks)

    # Remove grid and customize spines
    axes[i].grid(False)
    # For all but the last subplot: remove bottom spine and x-axis ticks
    if i < len(joint_names) - 1:
        sns.despine(ax=axes[i], top=True, right=True, bottom=True, left=False)
        axes[i].tick_params(bottom=False, labelbottom=False)
        # Add gap at bottom of left spine
        axes[i].spines["left"].set_position(("outward", 5))
    else:
        # For the last subplot: keep bottom spine and x-axis
        sns.despine(ax=axes[i], top=True, right=True, bottom=False, left=False)
        # Set x-axis spine to stop at the maximum time
        axes[i].spines["bottom"].set_bounds(0, max_time)
        # Add gap between left and bottom spines
        axes[i].spines["left"].set_position(("outward", 5))
        axes[i].spines["bottom"].set_position(("outward", 5))

    # Remove legends from all subplots
    if axes[i].get_legend():
        axes[i].get_legend().remove()

# Add shared y-axis label
fig.text(
    -0.02,
    0.5,
    "Joint Angle (degrees)",
    va="center",
    rotation="vertical",
    fontweight="bold",
)

legend_elements = [
    plt.Line2D([0], [0], color="orange", lw=2, label="STAC Registration"),
    plt.Line2D([0], [0], color="skyblue", lw=2, label="Track Replay"),
    Patch(facecolor="red", alpha=0.2, label="Difference"),
]
fig.legend(
    handles=legend_elements,
    loc="upper right",
    ncol=3,
    frameon=False,
    bbox_to_anchor=(1.08, 1.05),  # Fine-tune position
)

# Set common x-axis label
axes[-1].set_xlabel("Time (seconds)")

# Adjust layout to accommodate shared y-label and legend
plt.subplots_adjust(left=0.12, top=0.92)
plt.tight_layout()
plt.savefig(Path(ckpt_path) / "joint_angle_comparison_fly.pdf", dpi=150, bbox_inches="tight")
plt.show()

## Taking Advantage of Parallel Environment from JAX

JAX enables efficient parallel execution of multiple environment simulations simultaneously. This section demonstrates how to generate rollouts for multiple reference clips in parallel using JAX's `jit` (just-in-time compilation) and `vmap` (vectorized mapping) functions.

**Performance Note:** The parallel rollout generation leverages GPU/TPU acceleration to run multiple environments concurrently, significantly speeding up batch processing compared to sequential execution.

**Memory Considerations:** Adjust the `start_idx` and `end_idx` values based on your available system memory. If you encounter out-of-memory errors, reduce the batch size by narrowing the index range (e.g., process 50 clips at a time instead of 100).

In [ ]:
persistent_cache = True

# you can optionally choose to cache compiled computations to disk
if persistent_cache:
    jax.config.update("jax_compilation_cache_dir", "/tmp/jax_cache")
    jax.config.update("jax_persistent_cache_min_entry_size_bytes", -1)
    jax.config.update("jax_persistent_cache_min_compile_time_secs", 0)
    jax.config.update(
        "jax_persistent_cache_enable_xla_caches",
        "xla_gpu_per_fusion_autotune_cache_dir",
    )


jit_vmap_generate_rollout = jax.jit(jax.vmap(generate_rollout))

## Generate Parallel Rollouts

Generate rollouts for multiple reference clips in parallel using JAX's vectorized execution. This will create simulations for clips indexed from `start_idx` to `end_idx` (exclusive), running all environments simultaneously on GPU/TPU.

The resulting `batch_rollout` dictionary contains the same keys as a single rollout, but with an additional batch dimension corresponding to the number of clips processed in parallel.

In [ ]:
start_idx, end_idx = 0, 100
# this will generate rollouts for clip indices in [start_idx, end_idx)
clip_idxs = np.arange(start_idx, end_idx)
batch_rollout = jit_vmap_generate_rollout(clip_idxs)
print("These keys are available in batch_rollout:", list(batch_rollout.keys()))

## Batch Rollout Data Structure

The `batch_rollout` dictionary contains the same data as a single rollout, but with an additional batch dimension. The shape of the data is **(batch_dim, timestep, data_dim)**, where:

- **batch_dim**: Number of environments run in parallel (100 clips in this case, from index 0 to 99)
- **timestep**: Number of time steps in each rollout simulation
- **data_dim**: Dimension of the specific data type (e.g., qpos_dim for joint positions, action_dim for controls)

For example, `qposes_rollout` has shape `(100, timesteps, qpos_dim)`, containing the simulated joint positions for all 100 parallel rollouts.

In [ ]:
batch_rollout["qposes_rollout"].shape

In [ ]:
jnp.concat(batch_rollout["activations"]["intention"]).shape

## Neural Network Activation Analysis

This section explores the internal representations learned by the policy network. We'll use Principal Component Analysis (PCA) to visualize the high-dimensional activation space in 2D, revealing the latent structure of the policy's decision-making process throughout the rollout.

In [ ]:
# Extract intention activations (the latent representation before the policy head)
intention_activations = jnp.concat(batch_rollout["activations"]["intention"])
print(f"Intention activation shape: {intention_activations.shape}")
print(f"(timesteps, feature_dimensions)")

# Apply PCA to reduce to 3 dimensions
pca = PCA(n_components=3)
intention_pca = pca.fit_transform(intention_activations)

# Print explained variance
print(f"\nExplained variance ratio:")
print(f"PC1: {pca.explained_variance_ratio_[0]:.3f}")
print(f"PC2: {pca.explained_variance_ratio_[1]:.3f}")
print(f"PC3: {pca.explained_variance_ratio_[2]:.3f}")
print(f"Total: {pca.explained_variance_ratio_.sum():.3f}")

# Create time-based colormap
time_steps = np.arange(len(intention_pca))
time_seconds = time_steps * 0.01

# Create 3D plot
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection="3d")

# Plot trajectory with time-based color gradient
scatter = ax.scatter(
    intention_pca[:, 0],
    intention_pca[:, 1],
    intention_pca[:, 2],
    c=time_seconds,
    cmap="viridis",
    s=20,
    alpha=0.6,
    edgecolors="none",
)

# Add trajectory line to show temporal progression
ax.plot(
    intention_pca[:, 0],
    intention_pca[:, 1],
    intention_pca[:, 2],
    "gray",
    alpha=0.3,
    linewidth=1,
)

# Add colorbar
cbar = plt.colorbar(scatter, ax=ax, pad=0.1)
cbar.set_label("Time (seconds)", rotation=270, labelpad=20)

# Labels and title
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)", fontsize=12)
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)", fontsize=12)
ax.set_zlabel(f"PC3 ({pca.explained_variance_ratio_[2]:.1%} variance)", fontsize=12)
ax.set_title("PCA of Intention Space Activations (3D)", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.savefig(Path(ckpt_path) / "intention_pca_3d_fly.pdf", dpi=150, bbox_inches="tight")
plt.show()

## Exploring Rollout Data

The `batch_rollout` dictionary contains rich data from the parallel simulations that you can further analyze:

- **`qposes_rollout`** and **`qposes_ref`**: Simulated and reference joint positions for kinematic analysis
- **`ctrl`**: Motor control signals generated by the policy at each timestep
- **`activations`**: Internal neural network activations at different layers (encoder, intention, decoder, etc.)
- **`state_rewards`**: Reward values at each timestep, useful for evaluating policy performance

Feel free to explore these keys to gain deeper insights into the trained policy's behavior, decision-making process, and how well it tracks the reference motion.

Thank you for your interests in **mimic-mjx** !